# 🔬 Dataset 8: Breast Cancer Wisconsin (Original)
**Propósito:** Clasificación médica — Determinar si un tumor de mama es **Benigno (2)** o **Maligno (4)**.  
**Fuente:** Dr. William H. Wolberg, Universidad de Wisconsin (1992).  
**Método:** Aspiración con Aguja Fina (FNA) — evaluación citológica en escala 1–10.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos

In [ ]:
cols = ['Sample_ID','Clump_Thickness','Uniformity_Cell_Size','Uniformity_Cell_Shape',
        'Marginal_Adhesion','Single_Epithelial_Cell_Size','Bare_Nuclei',
        'Bland_Chromatin','Normal_Nucleoli','Mitoses','Class']

df = pd.read_csv('../datasets/4 data sets/dataset8/breast-cancer-wisconsin.data',
                 header=None, names=cols, na_values='?')

print(f"Shape: {df.shape}")
print(f"\nDistribución de clases:")
print(df['Class'].value_counts())
print("(2 = Benigno, 4 = Maligno)")
df.head(3)

## 2. La Variable Crítica: `Bare_Nuclei`
> Biológicamente, los núcleos desnudos deformados son un fuerte indicador de malignidad. Los 16 valores faltantes deben imputarse con cuidado clínico.

In [ ]:
print(f"Valores nulos en Bare_Nuclei: {df['Bare_Nuclei'].isnull().sum()}")
print(f"\nDistribución de Bare_Nuclei por clase:")
print(df.groupby('Class')['Bare_Nuclei'].describe())

In [ ]:
# Imputar con la mediana POR CLASE — respeta la lógica clínica
df['Bare_Nuclei'] = df.groupby('Class')['Bare_Nuclei'].transform(
    lambda x: x.fillna(x.median()))
print(f"Nulos restantes: {df.isnull().sum().sum()}")

## 3. EDA — Visualización Clínica

In [ ]:
features = ['Clump_Thickness','Uniformity_Cell_Size','Uniformity_Cell_Shape',
            'Marginal_Adhesion','Single_Epithelial_Cell_Size','Bare_Nuclei',
            'Bland_Chromatin','Normal_Nucleoli','Mitoses']

df['Diagnosis'] = df['Class'].map({2:'Benigno', 4:'Maligno'})

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for i, feat in enumerate(features):
    ax = axes[i//3][i%3]
    for label, group in df.groupby('Diagnosis'):
        group[feat].hist(ax=ax, alpha=0.6, bins=10, 
                         label=label, density=True,
                         color='steelblue' if label=='Benigno' else 'tomato')
    ax.set_title(feat)
    ax.legend(fontsize=7)
    ax.set_xlabel('Escala 1–10')

plt.suptitle('Distribución de 9 Atributos Citológicos — Benigno vs Maligno', 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('breast_eda.png', dpi=100)
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
corr = df[features + ['Class']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlación entre Atributos Citológicos')
plt.tight_layout()
plt.savefig('breast_corr.png', dpi=100)
plt.show()

## 4. Preparación

In [ ]:
df_model = df.drop(columns=['Sample_ID','Diagnosis'])
df_model['TARGET'] = (df_model['Class'] == 4).astype(int)  # 1=Maligno, 0=Benigno
df_model = df_model.drop(columns=['Class'])

X = df_model.drop(columns=['TARGET'])
y = df_model['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Prevalencia maligno en test: {y_test.mean():.1%}")

## 5. Modelos de Clasificación

In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)'          : SVC(probability=True, kernel='rbf', random_state=42)
}

for nombre, modelo in modelos.items():
    if 'Logística' in nombre or 'SVM' in nombre:
        modelo.fit(X_train_s, y_train)
        y_pred = modelo.predict(X_test_s)
        y_prob = modelo.predict_proba(X_test_s)[:,1]
    else:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_prob = modelo.predict_proba(X_test)[:,1]
    
    auc = roc_auc_score(y_test, y_prob)
    print(f"\n=== {nombre} | ROC-AUC: {auc:.4f} ===")
    print(classification_report(y_test, y_pred, target_names=['Benigno','Maligno']))

## 6. Matriz de Confusión — Relevancia Médica

In [ ]:
# En medicina: Falso Negativo (predecir benigno cuando es maligno) es MÁS GRAVE
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Benigno','Pred Maligno'],
            yticklabels=['Real Benigno','Real Maligno'])
plt.title('Matriz de Confusión — Random Forest\n(Falso Negativo = error más grave en medicina)')
plt.tight_layout()
plt.savefig('breast_confusion.png', dpi=100)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Verdaderos Negativos (Benigno correcto): {tn}")
print(f"Falsos Positivos    (Benigno→Maligno)  : {fp}")
print(f"Falsos Negativos    (Maligno→Benigno)  : {fn} ← ERROR CRÍTICO")
print(f"Verdaderos Positivos (Maligno correcto): {tp}")

## ✅ Resumen
| | |
|---|---|
| Registros | 699 |
| Benigno (2) | 458 (65.5%) |
| Maligno (4) | 241 (34.5%) |
| Mejor modelo | SVM / Random Forest |
| ROC-AUC aprox | ~0.99 |

**Consideración médica crítica:**  
El coste de un **Falso Negativo** (decir 'benigno' a un tumor maligno) es infinitamente mayor que el de un Falso Positivo. En producción se ajustaría el umbral de decisión por debajo de 0.5 para maximizar el Recall en clase Maligno.